In [44]:
from sklearn.metrics import roc_auc_score, mean_squared_error
from sklearn.preprocessing import LabelEncoder, MinMaxScaler

In [58]:
from deepctr_torch.inputs import SparseFeat, DenseFeat, get_feature_names

In [27]:
import random
import os
import pandas as pd
import numpy as np
ALL_GENRES = [
    'Action', 'Adventure', 'Animation', "Children's", 'Comedy',
    'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir',
    'Horror', 'Musical', 'Mystery', 'Romance', 'Sci-Fi',
    'Thriller', 'War', 'Western'
]

SEED = 1

def load_raw_data():
    ratings = pd.read_csv(
        'ratings.dat',
        sep='::', header=None, engine='python',
        names=['user_id', 'movie_id', 'rating', 'timestamp']
    )
    users = pd.read_csv(
        'users.dat',
        sep='::', header=None, engine='python',
        names=['user_id', 'gender', 'age', 'occupation', 'zip_code']
    )
    movies = pd.read_csv(
        'movies.dat',
        sep='::', header=None, engine='python',
        names=['movie_id', 'title', 'genres'],
        encoding='latin-1'
    )
    return ratings, users, movies


def build_genre_features(movies):
    """One-hot encode genres; returns movies df with binary genre columns."""
    for genre in ALL_GENRES:
        movies[genre] = movies['genres'].apply(
            lambda g: 1 if genre in g.split('|') else 0
        ).astype(np.int8)
    return movies


def build_negative_samples(ratings, users, movies_with_genres, neg_ratio=1):
    """
    For each positive (user, movie) pair, sample `neg_ratio` unobserved
    (user, movie) pairs as negatives.  Negatives get watch_label=0, rating=0.
    """
    user_ids = ratings['user_id'].unique()
    movie_ids = ratings['movie_id'].unique()
    positive_set = set(zip(ratings['user_id'], ratings['movie_id']))

    neg_rows = []
    total_neg = int(len(ratings) * neg_ratio)
    rng = np.random.default_rng(SEED)

    attempts = 0
    while len(neg_rows) < total_neg and attempts < total_neg * 20:
        u = rng.choice(user_ids)
        m = rng.choice(movie_ids)
        if (u, m) not in positive_set:
            neg_rows.append({'user_id': u, 'movie_id': m,
                             'rating': 0.0, 'timestamp': 0})
            positive_set.add((u, m))
        attempts += 1

    neg_df = pd.DataFrame(neg_rows)
    ratings['watch_label'] = 1
    neg_df['watch_label'] = 0

    combined = pd.concat([ratings, neg_df], ignore_index=True)
    # rating_label is the float rating (0 for negatives)
    combined['rating_label'] = combined['rating'].astype(float)
    return combined


def data_preparation():
    ratings, users, movies = load_raw_data()
    movies = build_genre_features(movies)

    df = build_negative_samples(ratings, users, movies)

    # Merge user and movie features
    df = df.merge(users, on='user_id', how='left')
    df = df.merge(
        movies[['movie_id'] + ALL_GENRES],
        on='movie_id', how='left'
    )

    target = ['watch_label', 'rating_label']

    # ------------------------------------------------------------------
    # Feature definitions
    # ------------------------------------------------------------------
    sparse_features = ['user_id', 'movie_id', 'gender', 'age', 'occupation']
    dense_features  = ['timestamp'] + ALL_GENRES  # genres as 0/1 dense cols

    # zip_code has too many values and mixed formats — encode as sparse
    df['zip_code'] = df['zip_code'].astype(str).str[:5]  # normalise length
    sparse_features.append('zip_code')

    # Label-encode sparse features
    for feat in sparse_features:
        lbe = LabelEncoder()
        df[feat] = lbe.fit_transform(df[feat].astype(str))

    # Scale dense features
    mms = MinMaxScaler(feature_range=(0, 1))
    df[dense_features] = mms.fit_transform(df[dense_features])

    # ------------------------------------------------------------------
    # Train / validation / test split  (60 / 20 / 20)
    # ------------------------------------------------------------------
    df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)
    n = len(df)
    train_df      = df.iloc[:int(n * 0.6)].reset_index(drop=True)
    validation_df = df.iloc[int(n * 0.6):int(n * 0.8)].reset_index(drop=True)
    test_df       = df.iloc[int(n * 0.8):].reset_index(drop=True)

    # ------------------------------------------------------------------
    # DeepCTR feature columns
    # ------------------------------------------------------------------
    fixlen_feature_columns = (
        [SparseFeat(feat,
                    vocabulary_size=int(df[feat].max()) + 1,
                    embedding_dim=4)
         for feat in sparse_features]
        + [DenseFeat(feat, 1) for feat in dense_features]
    )

    dnn_feature_columns    = fixlen_feature_columns
    linear_feature_columns = fixlen_feature_columns
    feature_names = get_feature_names(linear_feature_columns + dnn_feature_columns)

    train_model_input      = {name: train_df[name].values      for name in feature_names}
    validation_model_input = {name: validation_df[name].values for name in feature_names}
    test_model_input       = {name: test_df[name].values       for name in feature_names}

    return (
        train_df, train_model_input,
        validation_df, validation_model_input,
        test_df, test_model_input,
        dnn_feature_columns, target
    )

In [19]:
ratings, users, movies = load_raw_data()

In [20]:
ratings.head(5)

,user_id,movie_id,rating,timestamp
0,1,1193,5,978300760
1,1,661,3,978302109
2,1,914,3,978301968
3,1,3408,4,978300275
4,1,2355,5,978824291


In [21]:
users.head(5)

,user_id,gender,age,occupation,zip_code
0,1,F,1,10,48067
1,2,M,56,16,70072
2,3,M,25,15,55117
3,4,M,45,7,02460
4,5,M,25,20,55455


In [22]:
movies.head(5)

,movie_id,title,genres
0,1,Toy Story (1995),Animation|Children's|Comedy
1,2,Jumanji (1995),Adventure|Children's|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama
4,5,Father of the Bride Part II (1995),Comedy


In [23]:
movies = build_genre_features(movies)

In [24]:
movies.head(5)

,movie_id,title,genres,Action,Adventure,Animation,Children's,Comedy,Crime,Documentary,...,Fantasy,Film-Noir,Horror,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
0,1,Toy Story (1995),Animation|Children's|Comedy,0,0,1,1,1,0,0,...,0,0,0,0,0,0,0,0,0,0
1,2,Jumanji (1995),Adventure|Children's|Fantasy,0,1,0,1,0,0,0,...,1,0,0,0,0,0,0,0,0,0
2,3,Grumpier Old Men (1995),Comedy|Romance,0,0,0,0,1,0,0,...,0,0,0,0,0,1,0,0,0,0
3,4,Waiting to Exhale (1995),Comedy|Drama,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
4,5,Father of the Bride Part II (1995),Comedy,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0


In [28]:
df = build_negative_samples(ratings, users, movies)

In [33]:
df = df.merge(users, on='user_id', how='left')

In [36]:
df = df.merge(
    movies[['movie_id'] + ALL_GENRES],
    on='movie_id', how='left'
)

target = ['watch_label', 'rating_label']

In [38]:
sparse_features = ['user_id', 'movie_id', 'gender', 'age', 'occupation']
dense_features  = ['timestamp'] + ALL_GENRES  # genres as 0/1 dense cols

In [40]:
df['zip_code'] = df['zip_code'].astype(str).str[:5]  # normalise length
sparse_features.append('zip_code')

In [51]:
for feat in sparse_features:
    lbe = LabelEncoder()
    df[feat] = lbe.fit_transform(df[feat].astype(str))
    
mms = MinMaxScaler(feature_range=(0, 1))
df[dense_features] = mms.fit_transform(df[dense_features])

In [52]:
df.head(5)

,user_id,movie_id,rating,timestamp,watch_label,rating_label,gender,age,occupation,zip_code,...,Fantasy,Film-Noir,Horror,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
0,0,990,5.0,0.934872,1,5.0,0,0,12,640,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0,2640,3.0,0.934873,1,3.0,0,0,12,640,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0,2908,3.0,0.934873,1,3.0,0,0,12,640,...,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0
3,0,1673,4.0,0.934871,1,4.0,0,0,12,640,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0,418,5.0,0.935372,1,5.0,0,0,12,640,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [55]:
df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)
n = len(df)
train_df      = df.iloc[:int(n * 0.6)].reset_index(drop=True)
validation_df = df.iloc[int(n * 0.6):int(n * 0.8)].reset_index(drop=True)
test_df       = df.iloc[int(n * 0.8):].reset_index(drop=True)

In [59]:
fixlen_feature_columns = (
    [SparseFeat(feat,
                vocabulary_size=int(df[feat].max()) + 1,
                embedding_dim=4)
     for feat in sparse_features]
    + [DenseFeat(feat, 1) for feat in dense_features]
)

dnn_feature_columns    = fixlen_feature_columns
linear_feature_columns = fixlen_feature_columns
feature_names = get_feature_names(linear_feature_columns + dnn_feature_columns)

train_model_input      = {name: train_df[name].values      for name in feature_names}
validation_model_input = {name: validation_df[name].values for name in feature_names}
test_model_input       = {name: test_df[name].values       for name in feature_names}

In [74]:
pd.DataFrame(train_model_input).columns

Index(['user_id', 'movie_id', 'gender', 'age', 'occupation', 'zip_code',
       'timestamp', 'Action', 'Adventure', 'Animation', 'Children's', 'Comedy',
       'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror',
       'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War',
       'Western'],
      dtype='object')

In [73]:
pd.DataFrame(train_df[target].values)[1].value_counts()

1
0.0    599982
4.0    209745
3.0    156599
5.0    135691
2.0     64719
1.0     33514
Name: count, dtype: int64

In [71]:
print(train_df[target].dtypes)
print(train_df['rating_label'].value_counts())
print(train_df['watch_label'].value_counts())

watch_label       int64
rating_label    float64
dtype: object
rating_label
0.0    599982
4.0    209745
3.0    156599
5.0    135691
2.0     64719
1.0     33514
Name: count, dtype: int64
watch_label
1    600268
0    599982
Name: count, dtype: int64
